In [1]:
# Step 1: Install dependencies
!pip install transformers torch accelerate -q

In [2]:
# Step 2: Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [3]:
# Step 3: Function to query model
def ask_model(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):]


In [4]:
# Step 4: Test examples
prompts = [
    "What is the uncertainty principle in quantum mechanics?",
    "Solve: integrate x^2 dx from 0 to 3",
    "What is the eigenvalue of the Hamiltonian for a particle in a box?",
    "If F = ma, what is the acceleration of a 5kg object with 20N force?",
    "What is the Schrödinger equation?",
    "Solve: d/dx (sin(x^2))",
    "What is the speed of light in a medium with refractive index 1.5?",
    "What is the commutator [x, p] in quantum mechanics?",
    "Solve the differential equation: dy/dx = 2y",
    "What is the energy of a photon with wavelength 500nm?"
]


In [5]:
results = []
for prompt in prompts:
    output = ask_model(prompt)
    print(f"Q: {prompt}")
    print(f"A: {output}")
    print("-" * 50)
    results.append({"input": prompt, "model_output": output})

Q: What is the uncertainty principle in quantum mechanics?
A:  The uncertainty principle in quantum mechanics is a fundamental concept that describes the limitations on the precision with which certain pairs of physical properties of a particle, such as position and momentum, can be known simultaneously. It was first introduced by Werner Heisenberg in 1927 and is a cornerstone of quantum theory.

The uncertainty principle can be stated mathematically as follows: for any two observables \(A\) and \(B\) that are Hermitian operators (which correspond to physical properties like position and momentum), the product of the standard deviations of their measurements, \(\sigma_A\) and \(\sigma_B\), is bounded by:

\[
\sigma_A \sigma_B \geq \frac{\hbar}{2}
\]

where \(\hbar\) is the reduced Planck's constant, which is approximately \(1.0545718 \times 10^{-34} \, \text{J s}\).

This means that if you
--------------------------------------------------
Q: Solve: integrate x^2 dx from 0 to 3
A:  | W

In [8]:
!pip install huggingface_hub datasets -q
from huggingface_hub import notebook_login
notebook_login()

In [10]:
# Login to HuggingFace
from huggingface_hub import login
login(token="")

In [11]:
from datasets import Dataset

data = [
    {
        "input": "Solve: integrate x^2 dx from 0 to 3",
        "expected_output": "The integral of x^2 from 0 to 3 = [x^3/3] from 0 to 3 = 27/3 - 0 = 9",
        "model_output": "Returned HTML content from Wyzant tutoring website instead of solving",
        "error_type": "Web content hallucination"
    },
    {
        "input": "If F = ma, what is the acceleration of a 5kg object with 20N force?",
        "expected_output": "a = F/m = 20/5 = 4 m/s²",
        "model_output": "Returned HTML content from Quora instead of solving directly",
        "error_type": "Web content hallucination"
    },
    {
        "input": "Solve: d/dx (sin(x^2))",
        "expected_output": "Using chain rule: d/dx(sin(x^2)) = cos(x^2) * 2x = 2x*cos(x^2)",
        "model_output": "Returned HTML content from Study.com instead of solving",
        "error_type": "Web content hallucination"
    },
    {
        "input": "What is the speed of light in a medium with refractive index 1.5?",
        "expected_output": "v = c/n = (3×10^8)/1.5 = 2×10^8 m/s",
        "model_output": "Started correct derivation but stopped before computing the final numerical answer",
        "error_type": "Incomplete computation"
    },
    {
        "input": "What is the commutator [x, p] in quantum mechanics?",
        "expected_output": "[x, p] = iℏ",
        "model_output": "Explained the definition and steps but never stated the final result [x,p] = iℏ",
        "error_type": "Incomplete computation"
    },
    {
        "input": "Solve the differential equation: dy/dx = 2y",
        "expected_output": "y = Ce^(2x) where C is an arbitrary constant",
        "model_output": "Added initial condition y(0)=1 not present in the question, then solved only that specific case",
        "error_type": "Hallucinated problem constraints"
    },
    {
        "input": "What is the energy of a photon with wavelength 500nm?",
        "expected_output": "E = hc/λ = (6.626×10^-34 × 3×10^8)/(500×10^-9) = 3.976×10^-19 J ≈ 2.48 eV",
        "model_output": "Set up the formula correctly but stopped mid-calculation before computing the final answer",
        "error_type": "Incomplete computation"
    },
    {
        "input": "What is the uncertainty principle in quantum mechanics?",
        "expected_output": "Δx·Δp ≥ ℏ/2. Position and momentum cannot be simultaneously known with arbitrary precision.",
        "model_output": "Correct explanation but used general form σ_A·σ_B without specifying position-momentum first",
        "error_type": "Incomplete specificity"
    },
    {
        "input": "What is the eigenvalue of the Hamiltonian for a particle in a box?",
        "expected_output": "E_n = n²π²ℏ²/(2mL²) for n=1,2,3...",
        "model_output": "Correct formula given but response cut off mid-sentence without completing the explanation",
        "error_type": "Truncated response"
    },
    {
        "input": "What is the Schrödinger equation?",
        "expected_output": "iℏ ∂ψ/∂t = Ĥψ. Describes how quantum state evolves over time.",
        "model_output": "Correct equation but response truncated before completing the explanation",
        "error_type": "Truncated response"
    },
]

dataset = Dataset.from_list(data)
dataset.push_to_hub("Clambon/qwen2.5-1.5B-physics-math-blindspots")
print("Dataset uploaded successfully!")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 4.46kB / 4.46kB            

Dataset uploaded successfully!
